# Parkinson's Disease Detection From Vocal Biomarker
**Author:** Nasrul Hakim (nasrulhakim86)  
**Task:** Non-invasive neurological screening and classification of Parkinson's Disease (PD) using vocal acoustic measurements.  

---
### Overview & Clinical Background
Parkinson's disease is a neurodegenerative disorder affecting dopamine-producing neurons in the brain. Voice and speech impairments (dysarthria, hypophonia, monopitch, vocal tremor) affect up to 90% of individuals with Parkinson's disease and often appear in early stages.

This notebook trains and benchmarks machine learning models on vocal biomarkers (fundamental frequency variations, jitter, shimmer, harmonic-to-noise ratios, and nonlinear dynamic complexity metrics) to non-invasively classify Parkinson's disease from healthy controls.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')


## 2. Load and Explore the Dataset

In [ ]:
# Determine path to parkinsons.csv
candidates = ['parkinsons.csv', 'data/parkinsons.csv', '../data/parkinsons.csv', '../../parkinsons.csv']
dataset_path = next((p for p in candidates if os.path.exists(p)), 'parkinsons.csv')

df = pd.read_csv(dataset_path)
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
print('Missing values in columns:', df.isnull().sum().sum())
print('\nClass distribution (0: Healthy, 1: Parkinson\'s):')
print(df['status'].value_counts())


## 3. Exploratory Data Analysis & Biomarker Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.boxplot(x='status', y='MDVP:Fo(Hz)', data=df, ax=axes[0, 0], palette='Blues')
axes[0, 0].set_title('Fundamental Frequency (MDVP:Fo) vs Status')

sns.boxplot(x='status', y='MDVP:Jitter(%)', data=df, ax=axes[0, 1], palette='Oranges')
axes[0, 1].set_title('Jitter (%) vs Status')

sns.boxplot(x='status', y='MDVP:Shimmer', data=df, ax=axes[1, 0], palette='Greens')
axes[1, 0].set_title('Shimmer vs Status')

sns.boxplot(x='status', y='PPE', data=df, ax=axes[1, 1], palette='Purples')
axes[1, 1].set_title('Pitch Period Entropy (PPE) vs Status')
plt.tight_layout()
plt.show()

## 4. Preprocessing & Data Scaling

In [ ]:
# Drop name identifier column
X = df.drop(columns=['name', 'status'])
y = df['status']

feature_cols = list(X.columns)
print(f'Total feature count: {len(feature_cols)}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Training size: {X_train_scaled.shape}, Test size: {X_test_scaled.shape}')


## 5. Model Training and Benchmarking

In [ ]:
# 1. Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]

print('=== Random Forest Performance ===')
print(f'Accuracy:  {accuracy_score(y_test, rf_pred)*100:.2f}%')
print(f'Precision: {precision_score(y_test, rf_pred)*100:.2f}%')
print(f'Recall:    {recall_score(y_test, rf_pred)*100:.2f}%')
print(f'F1 Score:  {f1_score(y_test, rf_pred)*100:.2f}%')
print(f'ROC-AUC:   {roc_auc_score(y_test, rf_prob):.4f}')
print('\nClassification Report:\n', classification_report(y_test, rf_pred))


In [ ]:
# 2. Gradient Boosting Classifier
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
gb.fit(X_train_scaled, y_train)
gb_pred = gb.predict(X_test_scaled)
print('=== Gradient Boosting Performance ===')
print(f'Accuracy: {accuracy_score(y_test, gb_pred)*100:.2f}%')


## 6. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('Top 5 Most Discriminative Vocal Biomarkers:')
print(importances.head(5))

plt.figure(figsize=(10, 6))
importances.head(10).plot(kind='barh', color='teal').invert_yaxis()
plt.title('Top 10 Vocal Biomarkers by Relative Importance')
plt.xlabel('Importance')
plt.show()


## 7. Save Model Artifacts

In [ ]:
os.makedirs('output/models', exist_ok=True)
joblib.dump(rf, 'output/models/random_forest_model.pkl')
joblib.dump(scaler, 'output/models/scaler.pkl')
print('Successfully saved trained model and scaler to output/models/!')
